# Chomp 4xn Pattern Analysis
This notebook analyzes the P-positions output from the 4xn Chomp solver.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('p_positions_4xn_cpp.csv')
df['da'] = df['a'] - df['b']
df['db'] = df['b'] - df['c']
df['dc'] = df['c'] - df['d']

print(f"Loaded {len(df)} P-positions up to n={df['a'].max()}")

## Plot 1 - Difference Distributions

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

df['da'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Distribution of da (a-b)')
df['db'].value_counts().sort_index().head(30).plot(kind='bar', ax=axes[1], title='Distribution of db (b-c) (top 30)')
df['dc'].value_counts().sort_index().head(30).plot(kind='bar', ax=axes[2], title='Distribution of dc (c-d) (top 30)')

# Heatmap of (da, db) for top occurring pairs
heatmap_data = pd.crosstab(df['da'], df['db'])
# truncate for visibility
heatmap_data = heatmap_data.loc[:, heatmap_data.columns <= 20]
cax = axes[3].imshow(heatmap_data.values, cmap='YlOrRd', origin='lower', aspect='auto')
axes[3].set_title('Heatmap of (da) vs (db)')
axes[3].set_xlabel('db')
axes[3].set_ylabel('da')
fig.colorbar(cax, ax=axes[3])

plt.tight_layout()
plt.show()

## Plot 2 - Ratio Convergence

In [ ]:
df_ratios = df[df['a'] > 0].copy()
df_ratios['b/a'] = df_ratios['b'] / df_ratios['a']
df_ratios['c/a'] = df_ratios['c'] / df_ratios['a']
df_ratios['d/a'] = df_ratios['d'] / df_ratios['a']

plt.figure(figsize=(10, 6))
plt.scatter(df_ratios['a'], df_ratios['b/a'], alpha=0.1, s=1, label='b/a')
plt.scatter(df_ratios['a'], df_ratios['c/a'], alpha=0.1, s=1, label='c/a')
plt.scatter(df_ratios['a'], df_ratios['d/a'], alpha=0.1, s=1, label='d/a')

# Rolling means
a_vals = np.sort(df_ratios['a'].unique())
b_mean = [df_ratios[df_ratios['a'] == a]['b/a'].mean() for a in a_vals]
c_mean = [df_ratios[df_ratios['a'] == a]['c/a'].mean() for a in a_vals]
d_mean = [df_ratios[df_ratios['a'] == a]['d/a'].mean() for a in a_vals]

plt.plot(a_vals, b_mean, color='blue', label='b/a mean')
plt.plot(a_vals, c_mean, color='orange', label='c/a mean')
plt.plot(a_vals, d_mean, color='green', label='d/a mean')

plt.title('Ratios of dimensions relative to a')
plt.xlabel('a')
plt.ylabel('Ratio')
plt.legend()
plt.show()

## Plot 3 & Summary Table - Family Separation & Statistics

In [ ]:
df['family'] = list(zip(df['da'], df['db'], df['dc']))
family_counts = df['family'].value_counts()

summary = []

plt.figure(figsize=(12, 8))

for fam, count in family_counts.head(10).items():
    fam_df = df[df['family'] == fam]
    a_vals = fam_df['a'].values
    b_vals = fam_df['b'].values
    
    if len(a_vals) > 1:
        slope, intercept, r_value, _, _ = linregress(a_vals, b_vals)
        r_squared = r_value ** 2
    else:
        slope, intercept, r_squared = np.nan, np.nan, np.nan
        
    summary.append({
        'Family (da, db, dc)': fam,
        'Count': count,
        'Min_a': a_vals.min(),
        'Max_a': a_vals.max(),
        'Slope': slope,
        'Intercept': intercept,
        'R-squared': r_squared
    })
    
    plt.scatter(a_vals, b_vals, label=f'{fam} (R2={r_squared:.4f})', s=10)

plt.title('b vs a for Top 10 Families')
plt.xlabel('a')
plt.ylabel('b')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

summary_df = pd.DataFrame(summary)
display(summary_df)

## Plot 4 - Periodicity Check

In [ ]:
plt.figure(figsize=(12, 6))
for fam, count in family_counts.head(5).items():
    fam_df = df[df['family'] == fam].sort_values('a')
    a_vals = fam_df['a'].values
    diffs = np.diff(a_vals)
    plt.plot(diffs, marker='o', linestyle='none', label=str(fam), alpha=0.5)

plt.title('Step Sizes Between Consecutive a-values within Top Families')
plt.xlabel('Index in family')
plt.ylabel('Step size (a[i+1] - a[i])')
plt.legend()
plt.show()

print("If step sizes form a repeating pattern (horizontal repeating lines), the sequences are periodic.")

## Plot 5 - 3xn Boundary Overlay

In [ ]:
d0_df = df[df['d'] == 0]

plt.figure(figsize=(10, 10))
plt.scatter(d0_df['a'], d0_df['b'], c=d0_df['c'], cmap='viridis', label='4xn border (d=0) = 3xn', s=10)
plt.colorbar(label='c value')
plt.title('3xn P-positions (from 4xn dataset where d=0)')
plt.xlabel('a')
plt.ylabel('b')
plt.plot([0, 200], [0, 200], 'r--', alpha=0.5)  # y=x line
plt.show()

## Task 4: Conjecture Formalization

In [ ]:
N_THRESH = 150 # train on < 150, test on >= 150

train_df = df[df['a'] < N_THRESH]
test_df = df[df['a'] >= N_THRESH]

# Build predictive models for families with > 5 items
fam_models = {}
for fam, count in train_df['family'].value_counts().items():
    if count > 5:
        fam_df = train_df[train_df['family'] == fam]
        slope, intercept, r_value, _, _ = linregress(fam_df['a'], fam_df['b'])
        if r_value**2 > 0.99:
            fam_models[fam] = (slope, intercept)

print(f"Stated Conjecture: For sufficiently large n, every P-position of 4xn Chomp belongs to one of {len(fam_models)} families.")
print(f"Family rule: Given a, elements are (a, a - da, a - da - db, a - da - db - dc).")
print("Since a-b = da is exactly fixed for a family, b is explicitly a - da. The linear fit for b should trivially have slope=1, intercept=-da.")

# Evaluate coverage on test set
total_test = len(test_df)
predicted = 0

for _, row in test_df.iterrows():
    if tuple(row['family']) in fam_models:
        predicted += 1

print(f"Prediction Accuracy on a >= {N_THRESH}: {predicted} / {total_test} ({predicted/total_test*100:.2f}%)")

# --- Round 2: Deep Analysis ---
Based on the results from the initial survey, we are now re-parametrizing and investigating the growth and convergences precisely.

## Task 2: Re-parametrization & Task 3: Growth Analysis

In [ ]:
from fractions import Fraction

# Alternative 1: Ratio-based families (Exact Rationals)
def get_ratio_triple(row):
    a = row['a']
    return (Fraction(row['b'], a), Fraction(row['c'], a), Fraction(row['d'], a))

df['ratio_triple'] = df.apply(get_ratio_triple, axis=1)

# Growth Analysis
n_range = range(1, df['a'].max() + 1)
cum_da_families = []
cum_ratio_families = []
seen_da = set()
seen_ratio = set()

for n in n_range:
    n_df = df[df['a'] == n]
    seen_da.update(n_df['family'])
    seen_ratio.update(n_df['ratio_triple'])
    cum_da_families.append(len(seen_da))
    cum_ratio_families.append(len(seen_ratio))

plt.figure(figsize=(10, 5))
plt.plot(n_range, cum_da_families, label='Distinct (da, db, dc) Families')
plt.plot(n_range, cum_ratio_families, label='Distinct Ratio (b/a, c/a, d/a) Families')
plt.title('Cumulative Number of Distinct Families Encountered vs n')
plt.xlabel('n (a)')
plt.ylabel('Family Count')
plt.legend()
plt.grid(True)
plt.show()

print(f"Final da-family count (n={n}): {cum_da_families[-1]}")
print(f"Final ratio-family count (n={n}): {cum_ratio_families[-1]}")

### Task 2 Alternative 2: Geometric Ratio Families

In [ ]:
df_nonzero = df[(df['b'] > 0) & (df['c'] > 0)].copy()
df_nonzero['c_over_b'] = df_nonzero['c'] / df_nonzero['b']
df_nonzero['d_over_c'] = df_nonzero['d'] / df_nonzero['c']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df_nonzero['c_over_b'].plot(kind='hist', bins=100, ax=axes[0], title='Histogram of c/b')
df_nonzero['d_over_c'].plot(kind='hist', bins=100, ax=axes[1], title='Histogram of d/c')
plt.show()

print(f"Median c/b: {df_nonzero['c_over_b'].median():.6f}")
print(f"Median d/c: {df_nonzero['d_over_c'].median():.6f}")

### Task 2 Alternative 3: Polynomial Root Hypothesis

In [ ]:
import itertools

target_roots = [0.75, 0.45, 0.22] 

def check_polynomials(degree, coeff_range):
    coeffs_iter = itertools.product(range(-coeff_range, coeff_range + 1), repeat=degree + 1)
    best_p = None
    min_err = float('inf')
    
    for coeffs in coeffs_iter:
        if coeffs[0] == 0: continue
        p = np.poly1d(coeffs)
        roots = p.roots
        real_roots = roots[np.isreal(roots)].real
        
        if len(real_roots) >= 3:
            # Find best match for our targets
            sorted_roots = np.sort(real_roots)
            # We look for roots in (0, 1)
            valid_roots = sorted_roots[(sorted_roots > 0) & (sorted_roots < 1)]
            if len(valid_roots) >= 3:
                # Take top 3
                test_roots = valid_roots[-3:]
                err = np.sum((np.sort(test_roots) - np.sort(target_roots))**2)
                if err < min_err:
                    min_err = err
                    best_p = p
    return best_p, min_err

print("Searching for integer polynomials (degree <= 4, coeffs in [-2, 2])...")
p, err = check_polynomials(4, 2)
if p:
    print(f"Best Polynomial:\n{p}")
    print(f"Roots: {p.roots}")
    print(f"Error: {err}")

## Task 4: Exact Linearity & Slope Clustering

In [ ]:
slope_records = []
for fam in family_counts[family_counts >= 3].index:
    fam_df = df[df['family'] == fam].sort_values('a')
    p1 = fam_df.iloc[0]
    p2 = fam_df.iloc[1]
    
    da_val = p2['a'] - p1['a']
    db_val = p2['b'] - p1['b']
    slope = Fraction(db_val, da_val)
    
    # verify intercept is constant
    intercepts = [row['b'] - (slope * row['a']) for _, row in fam_df.iterrows()]
    is_constant = all(x == intercepts[0] for x in intercepts)
    
    slope_records.append({
        'family': fam,
        'slope': slope,
        'intercept': intercepts[0],
        'count': len(fam_df),
        'constant': is_constant
    })

slope_df = pd.DataFrame(slope_records)
print("Top Slopes by Frequency:")
display(slope_df['slope'].value_counts().head(10))

print(f"\nFamilies with non-constant linearity: {len(slope_df[~slope_df['constant']])}")

## Task 5: Convergence Rate Analysis

In [ ]:
from scipy.optimize import curve_fit

def convergence_model(a, L, K, p):
    return L + K * (a**(-p))

ratios = ['b/a', 'c/a', 'd/a']
results = {}

for r in ratios:
    # Windowed medians
    window_size = 10
    windows = range(10, 201, window_size)
    medians = []
    valid_a = []
    for w in windows:
        win_df = df_ratios[(df_ratios['a'] > w - window_size) & (df_ratios['a'] <= w)]
        if not win_df.empty:
            medians.append(win_df[r].median())
            valid_a.append(w)
    
    # Fit
    try:
        popt, _ = curve_fit(convergence_model, valid_a, medians, p0=[0.5, 1, 1], bounds=(0, [1, 10, 5]))
        results[r] = popt
        print(f"{r} Limit L: {popt[0]:.6f}, K: {popt[1]:.4f}, p: {popt[2]:.4f}")
    except:
        print(f"Could not fit {r}")

## Task 6: 3xn Cross-Dimensional Comparison

In [ ]:
d0 = df[df['d'] == 0].copy()
d0['prefix'] = list(zip(d0['a'], d0['b'], d0['c']))

extensions = {}
for prefix in d0['prefix']:
    ext_df = df[(df['a'] == prefix[0]) & (df['b'] == prefix[1]) & (df['c'] == prefix[2])]
    extensions[prefix] = ext_df['d'].tolist()

extension_counts = [len(v) for v in extensions.values()]
print(f"Max extensions per 3xn state: {max(extension_counts)}")
print(f"Fraction with unique extension: {extension_counts.count(1) / len(extension_counts):.2%}")

plt.hist(extension_counts, bins=range(1, 10))
plt.title('Number of d-extensions for each 3xn P-position')
plt.xlabel('Count of extensions')
plt.ylabel('Frequency')
plt.show()

# --- Round 3: Unique Extension & Algebraic Structure ---
This round rigorously tests the Bijective Lift conjecture and searches for the algebraic origins of the limit values.

## Task 1: Stress-Test the Unique Extension Conjecture

In [ ]:
# We need a baseline 3xn truth. We'll re-implement the tabulation logic here to compare.
def get_3xn_truth(max_n):
    p_positions = set([(1, 0, 0)])
    for a in range(1, max_n + 1):
        for b in range(a + 1):
            for c in range(b + 1):
                if (a, b, c) == (1, 0, 0): continue
                is_p = True
                # Row 2 moves
                for col in range(c):
                    if (a, b, col) in p_positions: is_p = False; break
                if not is_p: continue
                # Row 1 moves
                for col in range(b):
                    if (a, col, min(c, col)) in p_positions: is_p = False; break
                if not is_p: continue
                # Row 0 moves
                for col in range(1, a):
                    if (col, min(b, col), min(c, col)) in p_positions: is_p = False; break
                if is_p: p_positions.add((a, b, c))
    return p_positions

max_n_3 = df['a'].max()
print(f"Generating 3xn truth up to n={max_n_3}...")
truth_3 = get_3xn_truth(max_n_3)

prefix_groups = df.groupby(['a', 'b', 'c'])['d'].unique()
multi_ext = prefix_groups[prefix_groups.apply(len) > 1]

total_prefixes = len(prefix_groups)
one_ext = len(prefix_groups[prefix_groups.apply(len) == 1])

print(f"Total unique (a, b, c) prefixes in 4xn data: {total_prefixes}")
print(f"Prefixes with exactly 1 extension: {one_ext}")
print(f"Prefixes with 2+ extensions: {len(multi_ext)}")

if len(multi_ext) > 0:
    print("FAIL: Multi-extension cases found:")
    print(multi_ext)

prefixes_set = set(prefix_groups.index)
missing_3 = truth_3 - prefixes_set
print(f"3xn P-positions missing from 4xn prefixes: {len(missing_3)}")

if len(missing_3) > 0:
    print("FAIL: Missing 3xn extensions:")
    print(list(missing_3)[:10])

if len(multi_ext) == 0 and len(missing_3) == 0:
    print("VERDICT: PASS - 4xn is a Unique Bijective Lift of 3xn!")
else:
    print("VERDICT: FAIL - The conjecture does not hold exactly.")

## Task 2: Characterize d = f(a, b, c)

In [ ]:
from sklearn.linear_model import LinearRegression

plt.figure(figsize=(15, 4))
plt.hist(df['d'], bins=50, color='purple', alpha=0.7)
plt.title('Distribution of d values in 4xn P-positions')
plt.show()

corrs = df[['a', 'b', 'c', 'd']].corr()['d']
print("Correlation of d with row lengths:")
print(corrs)

X = df[['a', 'b', 'c']]
y = df['d']
reg = LinearRegression().fit(X, y)
print(f"\nMultivariate Fit: d = {reg.coef_[0]:.4f}*a + {reg.coef_[1]:.4f}*b + {reg.coef_[2]:.4f}*c + {reg.intercept_:.4f}")
print(f"R2 Score: {reg.score(X, y):.6f}")

# Minimal Input Tests
print(f"\nd determined uniquely by c? : {len(df.groupby('c')['d'].unique()) == df['c'].nunique()}")
print(f"d determined uniquely by (b, c)? : {len(df.groupby(['b', 'c'])['d'].unique()) == len(df.groupby(['b', 'c']))}")

## Task 3: Algebraic Search for Limit Values

In [ ]:
import math
import itertools

targets = [0.746653, 0.465585, 0.235482]
print("--- Task 3b: Polynomial Root Search (Degree <= 4) ---")

for deg in range(2, 5):
    found_this_deg = False
    for coeffs in itertools.product(range(-10, 11), repeat=deg):
        p_coeffs = [1] + list(coeffs)
        roots = np.roots(p_coeffs)
        reals = roots[np.isreal(roots)].real
        if all(any(abs(r - t) < 0.001 for r in reals) for t in targets):
            print(f"MATCH (deg {deg}): {p_coeffs} roots {sorted(reals)}")
            found_this_deg = True
    if found_this_deg: break

print("\n--- Task 3c: Constant Proximity Checks ---")
candidates = {
    'cos(3pi/7)': math.cos(3*math.pi/7),
    'cos(2pi/7)': math.cos(2*math.pi/7),
    'cos(pi/7)': math.cos(math.pi/7),
    'tribonacci': 0.74792,
    '1 - 1/e': 1 - 1/math.e,
    'ln(2)/ln(3)': math.log(2)/math.log(3),
}

for name, val in candidates.items():
    for i, t in enumerate(targets):
        if abs(val - t) < 0.01:
            print(f"CLOSE: {name} ({val:.6f}) matches L{i+1} ({t:.6f}) | err={abs(val-t):.6f}")

## Task 4: Recursive Lift Hypothesis Modeling

In [ ]:
bc_groups = df.groupby(['b', 'c', 'd'])['a'].unique()
reverse_unique = len(bc_groups[bc_groups.apply(len) == 1])
print(f"Is extension reversible? ((b,c,d) -> a unique): {reverse_unique / len(bc_groups):.2%}")

print(f"\n3xn P-position count (n={max_n_3}): {len(truth_3)}")
print(f"4xn P-position count (n={max_n_3}): {len(df)}")
if len(truth_3) == len(df):
    print("COUNT EQUALITY: Bijectivity holds exactly.")

# --- Round 4: The f(c) Function ---
Investigating the determinism d = f(c) and its periodic structure.

## Task 1 & 2: Visualization and Periodicity

In [ ]:
import scipy.signal

# Extract f(c)
fc_map = df.groupby('c')['d'].unique().apply(lambda x: x[0]).to_dict()
c_vals = sorted(fc_map.keys())
f_vals = [fc_map[c] for c in c_vals]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes[0, 0].plot(c_vals, f_vals)
axes[0, 0].set_title('f(c) vs c')

ratios = [f/c if c > 0 else 0 for f, c in zip(f_vals, c_vals)]
axes[0, 1].plot(c_vals, ratios)
axes[0, 1].set_title('f(c)/c vs c')

deltas = [f_vals[i] - f_vals[i-1] for i in range(1, len(f_vals))]
axes[1, 0].step(c_vals[1:], deltas)
axes[1, 0].set_title('First Differences (delta[c])')

second_deltas = [deltas[i] - deltas[i-1] for i in range(1, len(deltas))]
axes[1, 1].step(c_vals[2:], second_deltas)
axes[1, 1].set_title('Second Differences')
plt.tight_layout()
plt.show()

# Autocorrelation of deltas
plt.figure(figsize=(10, 4))
autocorr = scipy.signal.correlate(deltas, deltas, mode='full')
lags = np.arange(-len(deltas) + 1, len(deltas))
plt.plot(lags[len(lags)//2:], autocorr[len(autocorr)//2:])
plt.title('Autocorrelation of First Differences')
plt.xlabel('Lag')
plt.show()

## Task 3: Pattern matching & OEIS

In [ ]:
print("First 50 values of f(c):")
print(", ".join(map(str, f_vals[:50])))

print("\nFirst 50 deltas:")
print(", ".join(map(str, deltas[:50])))

L_est = results['d/a'][0] if 'd/a' in results else 0.235482
error_seq = [f - round(c * L_est) for f, c in zip(f_vals, c_vals)]
plt.figure(figsize=(10, 4))
plt.plot(c_vals, error_seq)
plt.title(f'Error sequence: f(c) - round(c * {L_est:.6f})')
plt.show()

## Task 4: Chain a -> b -> c -> d

In [ ]:
b_to_c = df.groupby('b')['c'].nunique()
a_to_b = df.groupby('a')['b'].nunique()

print(f"Fraction of b values with unique c: {(b_to_c == 1).mean():.2%}")
print(f"Fraction of a values with unique b: {(a_to_b == 1).mean():.2%}")
print(f"Max branching factor c per b: {b_to_c.max()}")
print(f"Max branching factor b per a: {a_to_b.max()}")

if (b_to_c == 1).all():
    print("CHAIN b -> c holds!")
else:
    print("CHAIN b -> c breaks.")

## Task 5: Prefix Coverage

In [ ]:
df['is_lifted'] = df.apply(lambda r: (r['a'], r['b'], r['c']) in truth_3, axis=1)
native_df = df[~df['is_lifted']]
lifted_df = df[df['is_lifted']]

print(f"Lifted (3xn P-prefix) count: {len(lifted_df)}")
print(f"Native (3xn N-prefix) count: {len(native_df)}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].scatter(lifted_df['b']/lifted_df['a'], lifted_df['c']/lifted_df['a'], alpha=0.1, label='Lifted')
axes[0].scatter(native_df['b']/native_df['a'], native_df['c']/native_df['a'], alpha=0.01, label='Native')
axes[0].set_title('Ratio plane: Native vs Lifted')
axes[0].legend()

native_df.groupby('c')['d'].nunique().value_counts().plot(kind='bar', ax=axes[1], title='Uniqueness count per c (Native only)')
plt.show()

# --- Round 5: Period-8 & Prefix Mask ---
This round focuses on identifying the algebraic and modular conditions of the prefix mask and validating the Period-8 signal.

## Task 1: Modular Residue Analysis of the Prefix Mask

In [ ]:
mod_results = []
for mod in [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 16]:
    for combo in ['a', 'b', 'c', 'a+b', 'b+c', 'a+c', 'a+b+c', 'a-b', 'b-c', 'a-c', 'a-b+c', 'a+b-c']:
        values = df.apply(lambda r: eval(combo, {'a': r['a'], 'b': r['b'], 'c': r['c']}), axis=1) % mod
        counts = values.value_counts(normalize=True).sort_index()
        uniformity = counts.std()
        if uniformity > 0.1:  # Higher std means less uniform, i.e., concentration
            mod_results.append({'combo': combo, 'mod': mod, 'counts': counts.to_dict(), 'std': uniformity})

mod_df = pd.DataFrame(mod_results).sort_values('std', ascending=False)
print("Top Non-Uniform Modular Residues:")
display(mod_df.head(10))

## Task 2: Connect Period-8 to 3xn Chomp Periodicity

In [ ]:
truth_3_list = sorted(list(truth_3))
c_truth = [p[2] for p in truth_3_list]
delta_c = np.diff(c_truth)

plt.figure(figsize=(10, 4))
autocorr_3 = scipy.signal.correlate(delta_c - np.mean(delta_c), delta_c - np.mean(delta_c), mode='full')
plt.plot(autocorr_3[len(autocorr_3)//2:])
plt.title('Autocorrelation of 3xn P-position c-sequence')
plt.show()

## Task 4: Prefix Mask as a Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

# Prepare dataset
all_triples = []
for a in range(1, 101): # Bounding for speed in n=200 subset
    for b in range(a + 1):
        for c in range(b + 1):
            all_triples.append({'a': a, 'b': b, 'c': c, 'label': int((a,b,c) in prefixes)})

mask_df = pd.DataFrame(all_triples)
mask_df['a_mod8'] = mask_df['a'] % 8
mask_df['b_mod8'] = mask_df['b'] % 8
mask_df['c_mod8'] = mask_df['c'] % 8
mask_df['sum_mod8'] = (mask_df['a'] + mask_df['b'] + mask_df['c']) % 8

X = mask_df[['a_mod8', 'b_mod8', 'c_mod8', 'sum_mod8']]
y = mask_df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

clf = DecisionTreeClassifier(max_depth=5)
clf.fit(X_train, y_train)
print(f"Classifier Accuracy (Modular Features): {clf.score(X_test, y_test):.4f}")
print(export_text(clf, feature_names=list(X.columns)))

## Task 5: Lexicographical Sorting & OEIS Prep

In [ ]:
lex_df = df.sort_values(['a', 'b', 'c'])
d_seq = lex_df['d'].tolist()
print(f"First 80 terms of d_seq (lexicographical): {d_seq[:80]}")
print(f"\nOEIS Search String: {', '.join(map(str, d_seq[:30]))}")